# NCF 구현하기 (3)
부제: NCF 공식 깃허브에 공개된 데이터를 사용하여 성능 확인 (ml-1m)

이전에 raw 데이터를 전처리하여 학습을 해보았는데, 성능이 안나옴. 뭐가 문젠지를 모르겠어서 헤매다가 데이터부터 바꿔보기로 함

In [1]:
import os, sys

import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf

sys.path.insert(0, '..')
from evaluate import ndcg, hr

2023-08-02 20:00:23.298280: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-08-02 20:00:23.489458: E tensorflow/stream_executor/cuda/cuda_blas.cc:2981] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2023-08-02 20:00:24.187892: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libnvinfer.so.7'; dlerror: libnvinfer.so.7: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64:/usr/local/cuda/lib64:/usr/local/nccl2/lib:/usr/local/cuda/extras/CUPTI/lib64
2023-08-02 20:00:24.187995: W tensorflow/stream_

# Data Load

데이터는 test_negative(99), test_positive(1), train_positive(n) 세 종류로 구성됨

In [2]:
movielens_dir = '../../neural_collaborative_filtering/Data/'

In [3]:
test_negative_df = pd.read_csv(os.path.join(movielens_dir, 'ml-1m.test.negative'), sep='\t', header=None)
print(test_negative_df.shape)
test_negative_df.head()

(6040, 100)


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
0,"(0,25)",1064,174,2791,3373,269,2678,1902,3641,1216,...,2854,3067,58,2551,2333,2688,3703,1300,1924,3118
1,"(1,133)",1072,3154,3368,3644,549,1810,937,1514,1713,...,1535,341,3525,1429,2225,1628,2061,469,3056,2553
2,"(2,207)",2216,209,2347,3,1652,3397,383,2905,2284,...,953,865,813,1353,2945,2580,2989,2790,2879,2481
3,"(3,208)",3023,1489,1916,1706,1221,1191,2671,81,2483,...,3347,1707,2901,2767,2167,1921,247,1618,2016,2323
4,"(4,222)",1794,3535,108,593,466,2048,854,1378,1301,...,2490,1332,2526,2804,2027,833,176,463,2851,2453


In [4]:
test_positive_df = pd.read_csv(os.path.join(movielens_dir, 'ml-1m.test.rating'), sep='\t', header=None)
print(test_positive_df.shape)
test_positive_df.head()

(6040, 4)


,0,1,2,3
0,0,25,5,978824351
1,1,133,3,978300174
2,2,207,4,978298504
3,3,208,4,978294282
4,4,222,2,978246585


In [5]:
train_positive_df = pd.read_csv(os.path.join(movielens_dir, 'ml-1m.train.rating'), sep='\t', header=None)
print(train_positive_df.shape)
train_positive_df.head()

(994169, 4)


,0,1,2,3
0,0,32,4,978824330
1,0,34,4,978824330
2,0,4,5,978824291
3,0,35,4,978824291
4,0,30,4,978824291


# 데이터 준비

모든 데이터가 이미 잘 준비되어 있으므로 간단한 처리만 하면 됨

train 데이터 정의 (positive only)

In [6]:
train_positive_df.iloc[:, 2] = 1
train_positive = train_positive_df.iloc[:, [0,1,2]].values
train_positive.shape

(994169, 3)

test 데이터 정의 (positive and negative, total 100)

In [7]:
test_data = []

# negative
test_negative = []
for i, row in test_negative_df.iloc[:, 1:].iterrows():
    test_negative.extend([[i,item,0] for item in row])

# positive
test_positive_df.iloc[:, 2] = 1
test_positive = test_positive_df.iloc[:, [0,1,2]].values

# test_data
test_data = np.concatenate([test_negative, test_positive])
test_data.shape

(604000, 3)

In [8]:
# np.random.shuffle(train_positive)
np.random.shuffle(test_data)

# Prep Data

- train negative sampling when training
- test data merge

In [9]:
# ## train negative sampling

# 전체 아이템 중에서 negative_num 만큼 샘플링하는데, 이 때 test 데이터나 train positive 데이터에 포함된 것으로 하면 안됨

In [10]:
# train_pos_array = train_positive.groupby(0)[1].apply(list).values
# test_pos_array = test_positive.groupby(0)[1].apply(list).values
# test_neg_array = test_negative.iloc[:, 1:].apply(list).values

# data_map = {i: np.concatenate([trainpos,testpos,testneg]) 
#     for i,(trainpos,testpos,testneg) in enumerate(zip(
#         train_pos_array, test_pos_array, test_neg_array))}

In [11]:
# def sampling(num_sample, all_items, without):
#     idx = 0
#     sampled = []
#     while idx < num_sample:
#         randint = np.random.choice(all_items)
#         if randint in without:
#             continue
#         else:
#             sampled.append(randint)
#             idx += 1
#     return sampled

In [12]:
# all_users = 6040
# all_items = 3706
# all_users_list = list(range(0, all_users))
# all_item_list = list(range(0, all_items))

In [13]:
# train_positive.iloc[:, 2] = 1
# train_data = [list(row) for row in train_positive.iloc[:, [0,1,2]].values]
# len(train_data)

In [14]:
# test_data = []

# for i in tqdm(all_users_list):
#     without = data_map[i]
#     neg_data = sampling(99, all_item_list, without)
#     # print([[i, neg, 0] for neg in neg_data])
#     test_data.extend([[i, neg, 0] for neg in neg_data])

# # train_data
# test_positive.iloc[:, 2] = 1
# test_data.extend([list(row) for row in test_positive.iloc[:, [0,1,2]].values])

# len(test_data)

In [15]:
# train = np.array(train_data)
# test = np.array(test_data)

In [16]:
# np.random.shuffle(train)
# np.random.shuffle(test)

# GMF (generalization matrix factorization)

## build

In [17]:
train_pos_array = train_positive_df.groupby(0)[1].apply(list).values
test_pos_array = test_positive_df.groupby(0)[1].apply(list).values
test_neg_array = test_negative_df.iloc[:, 1:].apply(list).values

data_map = {i: np.concatenate([trainpos,testpos,testneg]) 
    for i,(trainpos,testpos,testneg) in enumerate(zip(
        train_pos_array, test_pos_array, test_neg_array))}

In [18]:
# hyperparams
learning_rate = 0.001
batch_size = 256 #128, 256, 512, 1024
epochs = 20
num_neg = 4
topn = 10

In [19]:
all_users = 6040
all_items = 3706
all_users_list = list(range(0, all_users))
all_item_list = list(range(0, all_items))

In [20]:
def get_gmf(num_users, num_items, latent_dim):
    
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    user_flatten = tf.keras.layers.Flatten()(user_embedding)
    item_flatten = tf.keras.layers.Flatten()(item_embedding)

    x = tf.keras.layers.Lambda(lambda x: tf.multiply(x[0], x[1]))([user_flatten, item_flatten])
    x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.LecunNormal())(x)
    
    

    model = tf.keras.Model([user_input, item_input], x)
    return model

In [21]:
mirrored_strategy = tf.distribute.MirroredStrategy()

with mirrored_strategy.scope():
  gmf_model = get_gmf(all_users, all_items, 8)

gmf_model.summary()

2023-08-02 20:00:31.842912: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-08-02 20:00:31.844707: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-08-02 20:00:31.855832: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-08-02 20:00:31.857450: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-08-02 20:00:31.859038: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:980] successful NUMA node read from S

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Model: "model"
__________________________________________________________________________________________________
 Layer (type)                   Output Shape         Param #     Connected to                     
 input_1 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 input_2 (InputLayer)           [(None, 1)]          0           []                               
                                                                                                  
 user_embedding (Embedding)     (None, 1, 8)         48320       ['input_1[0][0]']                
                                                                                                  
 item_embedding (Embedding)     (None, 1, 8)

In [22]:
gmf_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="binary_crossentropy")

INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


## training

In [23]:
# negative sampling을 위한 함수
def sampling(num_sample, all_items, without):
    idx = 0
    sampled = []
    while idx < num_sample:
        randint = np.random.choice(all_items)
        if randint in without:
            continue
        else:
            sampled.append(randint)
            idx += 1
    return sampled

def prep_train_data(train_pos_array, num_neg):
    train_data = []

    for i in all_users_list:
        without = data_map[i]
        neg_data = sampling(num_neg, all_item_list, without)
        # print([[i, neg, 0] for neg in neg_data])
        train_data.extend([[i, neg, 0] for neg in neg_data])

    # train_data
    train_data = np.concatenate([train_data, train_pos_array])
    np.random.shuffle(train_data)
    return train_data

In [24]:
for iter in range(epochs):
    train_data = prep_train_data(train_positive, num_neg)

    gmf_history = gmf_model.fit(
        [train_data[:, 0], train_data[:, 1]], train_data[:, 2], epochs=1, 
        batch_size=batch_size)
    
    loss = gmf_model.evaluate([test_data[:, 0], test_data[:, 1]], test_data[:, 2], batch_size=2048, verbose=0)
    gmf_test_pred = gmf_model.predict([test_data[:, 0], test_data[:, 1]], batch_size=2048, verbose=0)

    ndcg_values = ndcg.average_ndcg(test_data[:, 0], gmf_test_pred, test_data[:, 2], topn, all_users)
    hr_values, hits = hr.average_hr(test_data[:, 0], gmf_test_pred, test_data[:, 2], topn, all_users)

    print(f"epoch: {iter}, loss: {loss}, HR: {hr_values}, NDCG: {ndcg_values}")

INFO:tensorflow:batch_all_reduce: 2 all-reduces with algorithm = nccl, num_packs = 1
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:GPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1').
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:GPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1').
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).
INFO:tensorflow:batch_all_reduce: 2 all-reduces with algorithm = nccl, num_packs = 1
INFO:tensorflow:Reduce to /job:local

In [26]:
# gmf_history = gmf_model.fit([train[:, 0], train[:, 1]], train[:, 2], epochs=10, 
#                             batch_size=batch_size, validation_split=0.1)

In [28]:
# import matplotlib.pyplot as plt

# plt.plot(gmf_history.history['loss'])
# # plt.plot(gmf_history.history['val_loss'])
# plt.show()

## prediction

In [30]:
gmf_test_pred = gmf_model.predict([test_data[:, 0], test_data[:, 1]], batch_size=2048)

295/295 [==============================] - 1s 3ms/step


In [32]:
# np.unique(test[:, 0], return_counts=True)

In [34]:
# 
ndcg_values = ndcg.average_ndcg(test_data[:, 0], gmf_test_pred, test_data[:, 2], topn, all_users)
hr_values, hits = hr.average_hr(test_data[:, 0], gmf_test_pred, test_data[:, 2], topn, all_users)

ndcg_values, hr_values

(0.25996742674621903, 0.47433774834437087)

## Multi Layer Perceptron

In [ ]:
def get_mlp(num_users, num_items, latent_dim, layer_dims):
    
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    user_flatten = tf.keras.layers.Flatten()(user_embedding)
    item_flatten = tf.keras.layers.Flatten()(item_embedding)

    x = tf.keras.layers.Concatenate()([user_flatten, item_flatten])

    # multi layers
    for i, l in enumerate(layer_dims):
        x = tf.keras.layers.Dense(units=l, activation="relu", dtype="float32", name=f'{i}st_layer',
            kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    # last layer
    x = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)

    model = tf.keras.Model([user_input, item_input], x)
    return model

In [ ]:
mlp_model = get_mlp(all_users, all_items, 16, [32, 16, 8])
mlp_model.summary()

In [ ]:
learning_rate = 0.001
mlp_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate), loss="binary_crossentropy")#, metrics=['acc'])

In [ ]:
mlp_history = mlp_model.fit(
    [train[:, 0], train[:, 1]], train[:, 2], epochs=50, batch_size=batch_size, validation_split=0.1)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(mlp_history.history['loss'])
plt.plot(mlp_history.history['val_loss'])
plt.show()

In [ ]:
mlp_test_pred = mlp_model.predict([test[:, 0], test[:, 1]], batch_size=2048)

# NeuMF (Neural Matrix Factorization)

In [ ]:
# mlp
ndcg_values = ndcg.average_ndcg(test[:, 0], mlp_test_pred, test[:, 2], topn, all_users)
hr_values, hits = hr.average_hr(test[:, 0], mlp_test_pred, test[:, 2], topn, all_users)

ndcg_values, hr_values

# Evaluation

In [ ]:
def get_neuMF(num_users, num_items, latent_dim, layer_dims):

    # common inputs
    user_input = tf.keras.layers.Input(shape=1, dtype="float32")
    item_input = tf.keras.layers.Input(shape=1, dtype="float32")

    # gmf branch
    gmf_user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='gmf_user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    gmf_item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='gmf_item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    gmf_user_flatten = tf.keras.layers.Flatten()(gmf_user_embedding)
    gmf_item_flatten = tf.keras.layers.Flatten()(gmf_item_embedding)

    gmf_multiply = tf.keras.layers.Lambda(lambda x: tf.multiply(x[0], x[1]), name='gmf_multiply')(
        [gmf_user_flatten, gmf_item_flatten])

    # mlp branch
    mlp_user_embedding = tf.keras.layers.Embedding(input_dim=num_users, output_dim=latent_dim, name='mlp_user_embedding',
                    embeddings_initializer='normal', input_length=1)(user_input)
    mlp_item_embedding = tf.keras.layers.Embedding(input_dim=num_items, output_dim=latent_dim, name='mlp_item_embedding',
                    embeddings_initializer='normal', input_length=1)(item_input)
    
    mlp_user_flatten = tf.keras.layers.Flatten()(mlp_user_embedding)
    mlp_item_flatten = tf.keras.layers.Flatten()(mlp_item_embedding)

    x = tf.keras.layers.Concatenate()([mlp_user_flatten, mlp_item_flatten])

    # multi layers
    for i, l in enumerate(layer_dims):
        x = tf.keras.layers.Dense(units=l, activation="relu", dtype="float32", name=f'{i}st_layer',
            kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42))(x)
        
    # concat branches
    x = tf.keras.layers.Concatenate()([gmf_multiply, x])

    # last layer
    neumf_last_layer = tf.keras.layers.Dense(units=1, activation="sigmoid", dtype="float32", name='last_layer',
        kernel_initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=0.01, seed=42), bias_constraint=None)(x)

    neumf_model = tf.keras.Model([user_input, item_input], neumf_last_layer)
    return neumf_model


In [ ]:
neuMF_model = get_neuMF(len(all_users), len(all_items), 16, [32, 16, 8])
neuMF_model.summary()

## load saved weights

In [ ]:
learning_rate = 0.01
neuMF_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate, validation_split=0.1)

## Training

In [ ]:
neuMF_history = neuMF_model.fit([train[:, 0], train[:, 1]], train[:, 2], epochs=150, batch_size=2048)

In [ ]:
import matplotlib.pyplot as plt

plt.plot(neuMF_history.history['loss'])
plt.show()

In [ ]:
neuMF_test_pred = neuMF_model.predict([test[:, 0], test[:, 1]], batch_size=2048)

In [ ]:
# neuMF
ndcg_values = ndcg.average_ndcg(test[:, 0], neuMF_test_pred, test[:, 2], topn, len(all_users))
hr_values, hits = hr.average_hr(test[:, 0], neuMF_test_pred, test[:, 2], topn, len(all_users))

ndcg_values, hr_values